# GA RF method <img align="right" src="../Supplementary_data/dea_logo.jpg">

* [**Sign up to the DEA Sandbox**](https://docs.dea.ga.gov.au/setup/sandbox.html) to run this notebook interactively from a browser
* **Compatibility:** Notebook currently compatible with specific `DEA Dev Sandbox` environment: HNRS test
* **Products used:** 
[ga_ls8c_nbart_gm_4fyear_3](http://dea-public-data-dev.s3-website-ap-southeast-2.amazonaws.com/?prefix=projects/burn_cube/)
[ga_ls8c_ard_3](https://explorer.sandbox.dea.ga.gov.au/ga_ls8cls9c_gm_cyear_3)
* **Special requirements:** run in the HNRS test DEA dev sandbox environment
* **Prerequisites:** An _optional_ list of any notebooks that should be run or content that should be understood prior to launching this notebook

## Getting started

To run this notebook you need to be in a HNRS sandbox environment, so that you can access the 4 year geomedians which are not stored in a public folder as they are an intermediate product purely for running analysis.

In [35]:
import logging
import os
import re
import sys
from typing import Tuple

import click
import datacube
import numpy as np
import requests
import rioxarray
import xarray as xr
from datacube.utils.cog import write_cog
from dea_tools.bandindices import calculate_indices
from dea_tools.classification import predict_xr
from joblib import load
from odc.dscache.tools.tiling import parse_gridspec_with_name
from scipy import ndimage
from scipy.ndimage._measurements import _stats
from skimage import morphology
from skimage.segmentation import quickshift

In [2]:
def _get_gpgon(
    region_id: str,
) -> Tuple[datacube.utils.geometry.Geometry, datacube.utils.geometry._base.GeoBox]:
    """
    Get a geometry that covers the specified region for use with datacube.load().

    Parameters
    ----------
    region_id : str
        The ID of the region to get a geometry for. E.g. x30y29

    Returns
    -------
    Tuple[datacube.utils.geometry.Geometry, datacube.utils.geometry._base.GeoBox]
        The geometry object representing the region specified by `region_id` and the corresponding geobox.
    """

    _, gridspec = parse_gridspec_with_name("au-30")

    # gridspec : au-30
    pattern = r"x(\d+)y(\d+)"

    match = re.match(pattern, region_id)

    x = int(match.group(1))
    y = int(match.group(2))

    geobox = gridspec.tile_geobox((x, y))

    # Return the resulting Geometry object
    return datacube.utils.geometry.Geometry(geobox.extent.geom, crs="epsg:3577"), geobox

In [3]:
# Define the feature_layers function
# This function generates the data required by the RF model to map burnt area
def feature_layers(
    ard_query, gm_query, hnrs_dc, dc, time_pre, time_post, climate_dataset, pre_fire_gm_product_name
):
    # Load ls8 4-year geomedian for the specified time period and query parameters
    ds_base = hnrs_dc.load(
        # product="ga_ls8c_nbart_gm_4cyear_3",
        product=pre_fire_gm_product_name,
        # time=("2017-01-01", "2017-12-31"),  # calendar year
        time=time_pre,  # calendar year
        **ard_query,
    )

    # Load post-fire annual geomedian
    ds_post = dc.load("ga_ls8cls9c_gm_cyear_3", time=time_post, **gm_query)

    # Load Land Cover
    # NOTE: the ga_ls_landcover_class_fyear_3 is old Collection 3 LC, will chnage it 
    # in the future
    lc_query = ard_query
    lc_query["measurements"] = ["level3", "level4"]

    ds_lc = dc.load("ga_ls_landcover_class_fyear_3", time=time_post, **lc_query)

    # the landcover level 3 and level 4 should convert to one-hot encoding data.
    
    # level 3
    # 0: No data
    # 111: Cultivated Terrestrial Vegetation (CTV)
    # 112: (Semi-)Natural Terrestrial Vegetation (NTV)
    # 124: Natural Aquatic Vegetation (NAV)
    # 215: Artificial Surface (AS)
    # 216: Natural Bare Surface (NS)
    # 220: Water

    for level3_key in [0, 111, 112, 124, 215, 216, 220]:
        level3_key_name = f"level3_{str(level3_key)}"
        ds_lc[level3_key_name] = xr.where(ds_lc["level3"] == level3_key, 1, 0)

    # Drop the original 'level3' variable
    ds_lc = ds_lc.drop_vars("level3")

    # level 4
    # refs to detail table here: https://knowledge.dea.ga.gov.au/data/product/dea-land-cover-landsat/?tab=details

    level4_keys = [0, 1, 3, 4, 5, 6, 7, 8, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 
                    25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 55, 56, 57, 58, 59, 
                    60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 
                    77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 
                    94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104]

    for level4_key in level4_keys:
        level4_key_name = f"level4_{str(level4_key)}"
        ds_lc[level4_key_name] = xr.where(ds_lc["level4"] == level4_key, 1, 0)

    # Drop the original 'level4' variable
    ds_lc = ds_lc.drop_vars("level4")

    ds_lc = ds_lc.isel(time=0)
    ds_lc = ds_lc.drop_vars("time")

    ds_base = ds_base.load()
    
    
    # Calculate band indices for pre and post-fire data
    # Calculate the base(pre) indices
    da_base = calculate_indices(
        ds_base, index=["NDVI", "NBR", "NDMI"], drop=False, collection="ga_gm_3"
    )
    da_base["VARI_pre"] = (ds_base.green - ds_base.red) / (
        ds_base.green + ds_base.red - ds_base.blue
    )

    # Renaming these indices for clarity with '_pre' suffixes
    da_base = da_base.rename({"NDVI": "NDVI_pre", "NDMI": "NDMI_pre", "NBR": "NBR_pre"})

    # Calculate the post indices
    da_post = calculate_indices(
        ds_post, index=["NDVI", "NBR", "NDMI"], drop=False, collection="ga_ls_3"
    )
    da_post["VARI_post"] = (ds_post.nbart_green - ds_post.nbart_red) / (
        ds_post.nbart_green + ds_post.nbart_red - ds_post.nbart_blue
    )
    da_post["BAI_post"] = 1 / (((0.1 - ds_post.nbart_red) ** 2) + ((0.06 - ds_post.nbart_nir) ** 2))
    # Renaming these indices for clarity with '_post' suffixes
    da_post = da_post.rename(
        {"NDVI": "NDVI_post", "NDMI": "NDMI_post", "NBR": "NBR_post"}
    )

    # Calculate differences in some indices between pre and post-fire data
    dndvi = da_base.NDVI_pre.isel(time=0) - da_post.NDVI_post
    dndvi = dndvi.rename("dNDVI")
    dnbr = da_base.NBR_pre.isel(time=0) - da_post.NBR_post
    dnbr = dnbr.rename("dNBR")
    dndmi = da_base.NDMI_pre.isel(time=0) - da_post.NDMI_post
    dndmi = dndmi.rename("dNDMI")
    dvari = da_base.VARI_pre.isel(time=0) - da_post.VARI_post
    dvari = dvari.rename("dVARI")

    # Remove unnecessary variables from the datasets
    drop_list = ["green", "red", "blue", "nir", "swir1", "swir2"]
    da_base = da_base.drop_vars(drop_list)
    da_base = da_base.isel(time=0)
    da_base = da_base.drop_vars("time")
    da_post = da_post.drop_vars(["nbart_blue", "nbart_green", "nbart_red", "nbart_nir", "nbart_swir_1", "nbart_swir_2"])

    # Extract climate data based on the specified geographical polygon (query_pgon)
    query_pgon = ard_query["geopolygon"]
    x_range = query_pgon.boundingbox.range_x
    y_range = query_pgon.boundingbox.range_y
    ds_climate = climate_dataset.sel(
        x=slice(x_range[0], x_range[1]), y=slice(y_range[1], y_range[0])
    )

    # Reproject the climate data to match the post-fire data's CRS
    da_post = da_post.rio.write_crs("EPSG:3577")
    ds_climate = ds_climate.rio.write_crs("EPSG:3577")
    ds_climate = ds_climate.rio.reproject_match(da_post)

    # Create new climate code variables based on the value of 'climate_code'
    ds_climate["climate_code_1"] = ds_climate["climate_code"] * 0
    ds_climate["climate_code_2"] = ds_climate["climate_code"] * 0
    ds_climate["climate_code_3"] = ds_climate["climate_code"] * 0

    ds_climate["climate_code_1"] = xr.where(
        ds_climate["climate_code"] == 1, 1, ds_climate["climate_code_1"]
    )
    ds_climate["climate_code_2"] = xr.where(
        ds_climate["climate_code"] == 2, 1, ds_climate["climate_code_2"]
    )
    ds_climate["climate_code_3"] = xr.where(
        ds_climate["climate_code"] == 3, 1, ds_climate["climate_code_3"]
    )

    # Drop the original 'climate_code' variable
    ds_climate = ds_climate.drop_vars("climate_code")

    # Merge all the datasets into a single result dataset
    result = xr.merge(
        [da_post, da_base, ds_lc, dnbr, dndvi, dvari, dndmi, ds_climate], compat="override"
    )

    return result

In [4]:
region_id = "x14y32"

In [5]:
dc = datacube.Datacube(
    app=f"Burn Cube K8s processing - {region_id}",
    config={
            "db_hostname": os.getenv("DB_HOSTNAME"),
            "db_password": os.getenv("ODC_DB_PASSWORD"),
            "db_username": os.getenv("ODC_DB_USERNAME"),
            "db_port": 5432,
            "db_database": os.getenv("ODC_DB_DATABASE"),
    },
)

hnrs_dc = datacube.Datacube(
    app=f"Burn Cube K8s processing - {region_id}",
    config={
        "db_hostname": os.getenv("DB_HOSTNAME"),
        "db_password": os.getenv("HNRS_DC_DB_PASSWORD"),
        "db_username": os.getenv("HNRS_DC_DB_USERNAME"),
        "db_port": 5432,
        "db_database": os.getenv("DB_DATABASE"),},
)

# need to set the AWS login so that we can access the data we need
os.environ["AWS_NO_SIGN_REQUEST"] = "Yes"

In [6]:
process_cfg_url = "https://raw.githubusercontent.com/GeoscienceAustralia/burn-mapping/develop/configs/vic_rf_processings/ga_ls8c_nbart_vic_xgb_cyear_3.yaml"

In [7]:
from typing import Any, Dict, Tuple
import fsspec
import yaml

def load_yaml_remote(yaml_url: str) -> Dict[str, Any]:
    """
    Open a yaml file remotely and return the parsed yaml document
    """
    with fsspec.open(yaml_url, mode="r") as f:
        return next(yaml.safe_load_all(f))

In [8]:
process_cfg = load_yaml_remote(process_cfg_url)

pre_fire_gm_product_name = process_cfg["input_products"]["geomed_name"]
output_folder = process_cfg["output_folder"]
time_pre = ("2017-01-01", "2017-12-31")
feature_list = process_cfg["model_features"]

output_product_name = process_cfg["product"]["name"]

In [9]:
climate_dataset = rioxarray.open_rasterio("remapped_koppen_data_3classes_3577.tif")

In [10]:
climate_dataset = climate_dataset.to_dataset("band")

In [11]:
# Rename variable 1 to 'climate_code' for clarity and easier access.
climate_dataset = climate_dataset.rename({1: "climate_code"})

climate_dataset = climate_dataset.where(
        climate_dataset["climate_code"] != 2147483647
)

In [12]:
model_path = "dea_ml_ba_xgb_with_landcover_rf_model_01_08_2024.joblib"

In [13]:
# Load the machine learning model from the specified file using the `load` function from the `joblib` library.
model = load(model_path)

In [14]:
box = _get_gpgon(region_id)
pgon = box[0]  # it always only one polygon there

In [15]:
# Define the resolution of the geospatial data.
resolution = (-30, 30)

# Define the output coordinate reference system (CRS).
output_crs = "epsg:3577"

# Define a list of bands to load
measurements = ["blue", "green", "red", "nir", "swir1", "swir2"]

gm_measurements = ["nbart_blue", "nbart_green", "nbart_red", "nbart_nir", "nbart_swir_1", "nbart_swir_2"]

# Define the analysis year
time_post = "2020"

# Create a dictionary query object to pass to the `feature_layers` function
ard_query = {
        "resolution": resolution,
        "output_crs": output_crs,
        "measurements": measurements,
        "geopolygon": pgon,
    }

gm_query = {
        "resolution": resolution,
        "output_crs": output_crs,
        "measurements": gm_measurements,
        "geopolygon": pgon,
}

data = feature_layers(
        ard_query,
        gm_query,
        hnrs_dc,
        dc,
        time_pre,
        time_post,
        climate_dataset,
        pre_fire_gm_product_name,
    ).squeeze()


# this can make sure no issue on feature name order
reorder_data = data[feature_list]

predicted = predict_xr(
        model, reorder_data, proba=True, persist=True, clean=True, return_input=True
    ).compute()

x_range = pgon.boundingbox.range_x
y_range = pgon.boundingbox.range_y


predicting...
   probabilities...
   input features...


In [16]:
# Load the water observations data over the processed tile and analysis year
wo = dc.load(
        product="ga_ls_wo_fq_cyear_3",
        crs="EPSG:3577",
        output_crs="EPSG:3577",
        x=x_range,
        y=y_range,
        time="2020",
    )

In [17]:
# Create water mask to mask pixels that have more than 20% wet observations
# Plot the water mask
wo_mask = wo.frequency > 0.2

predicted_wofs = xr.where(wo_mask == 0, predicted, 0)

In [18]:
predicted_wofs

<xarray.Dataset> Size: 9GB
Dimensions:         (time: 1, y: 3200, x: 3200)
Coordinates:
  * time            (time) datetime64[ns] 8B 2020-07-01T23:59:59.999999
  * y               (y) float64 26kB -2.304e+06 -2.304e+06 ... -2.4e+06 -2.4e+06
  * x               (x) float64 26kB -1.344e+06 -1.344e+06 ... -1.248e+06
    spatial_ref     int32 4B 3577
Data variables: (12/106)
    Predictions     (time, y, x) int64 82MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0
    Probabilities   (time, y, x) float32 41MB 820.2 724.2 899.0 ... 463.0 742.1
    NDVI_post       (time, y, x) float64 82MB 0.3084 0.3074 ... 0.2064 0.1805
    NBR_post        (time, y, x) float64 82MB 0.05604 0.05768 ... 0.039 0.007168
    NDMI_post       (time, y, x) float64 82MB -0.08081 -0.08386 ... -0.1347
    VARI_post       (time, y, x) float64 82MB -0.2379 -0.2369 ... -0.2887
    ...              ...
    level4_99       (time, y, x) float64 82MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    level4_100      (time, y, x) float64 82MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    level4_101      (time, y, x) float64 82MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    level4_102      (time, y, x) float64 82MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    level4_103      (time, y, x) float64 82MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    level4_104      (time, y, x) float64 82MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0

In [19]:
# Define the size of the disk structuring element, measured in number of pixels.
# The default value is 2.
disk_size = 2

# Remove the time index from the xr dataarray
all_burn = predicted_wofs.Predictions.isel(time=0)

# Perform an opening morphological operation on the `all_burn` dataarray
opened_data = xr.DataArray(
        morphology.binary_opening(all_burn, morphology.disk(disk_size)),
        coords=all_burn.coords,
    )
# Perform a closing morphological operation on the `opened_data` dataarray
dilated_data = xr.DataArray(
        ndimage.binary_dilation(opened_data, morphology.disk(disk_size + 1)),
        coords=all_burn.coords,
)

# Set the post-processed data to the `all_burn_cleaned` variable, and convert to a float dtype
all_burn_cleaned = dilated_data
all_burn_cleaned = all_burn_cleaned.astype(int)
all_burn_cleaned = all_burn_cleaned.astype("float64")

# Reapply the wo mask, to remove burnt pixels over water bodies that the above closing created
all_burn_cleaned = xr.where(wo_mask == 0, all_burn_cleaned, 0)

# Ensure the crs attribute is set to 3577 using the wo dc
all_burn_cleaned.attrs["crs"] = wo.crs

nm_xy = region_id  # dynamic build from data loading process
nm_date = "2020"  # see what is in bc, based upon nm_yeartype decision from above

In [20]:
all_burn_cleaned

<xarray.DataArray (time: 1, y: 3200, x: 3200)> Size: 82MB
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]]])
Coordinates:
  * time         (time) datetime64[ns] 8B 2020-07-01T23:59:59.999999
  * y            (y) float64 26kB -2.304e+06 -2.304e+06 ... -2.4e+06 -2.4e+06
  * x            (x) float64 26kB -1.344e+06 -1.344e+06 ... -1.248e+06
    spatial_ref  int32 4B 3577
Attributes:
    crs:      epsg:3577

In [21]:
all_burn_ds = all_burn_cleaned.to_dataset(name="all_burn_ds")

In [22]:
all_burn_ds

<xarray.Dataset> Size: 82MB
Dimensions:      (time: 1, y: 3200, x: 3200)
Coordinates:
  * time         (time) datetime64[ns] 8B 2020-07-01T23:59:59.999999
  * y            (y) float64 26kB -2.304e+06 -2.304e+06 ... -2.4e+06 -2.4e+06
  * x            (x) float64 26kB -1.344e+06 -1.344e+06 ... -1.248e+06
    spatial_ref  int32 4B 3577
Data variables:
    all_burn_ds  (time, y, x) float64 82MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0

In [23]:
all_burn_ds = all_burn_ds.isel(time=0, drop=True)

In [24]:
all_burn_ds["all_burn_ds"]

<xarray.DataArray 'all_burn_ds' (y: 3200, x: 3200)> Size: 82MB
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])
Coordinates:
  * y            (y) float64 26kB -2.304e+06 -2.304e+06 ... -2.4e+06 -2.4e+06
  * x            (x) float64 26kB -1.344e+06 -1.344e+06 ... -1.248e+06
    spatial_ref  int32 4B 3577
Attributes:
    crs:      epsg:3577

In [25]:
all_burn_da = all_burn_ds["all_burn_ds"]

In [26]:
all_burn_da

<xarray.DataArray 'all_burn_ds' (y: 3200, x: 3200)> Size: 82MB
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])
Coordinates:
  * y            (y) float64 26kB -2.304e+06 -2.304e+06 ... -2.4e+06 -2.4e+06
  * x            (x) float64 26kB -1.344e+06 -1.344e+06 ... -1.248e+06
    spatial_ref  int32 4B 3577
Attributes:
    crs:      epsg:3577

In [27]:
write_cog(geo_im=all_burn_da, fname="test.tif", overwrite=True, nodata=-999)

PosixPath('test.tif')

In [28]:
all_burn_da.dims

('y', 'x')

In [29]:
all_burn_cleaned = all_burn_cleaned.isel(time=0)

In [30]:
all_burn_cleaned

<xarray.DataArray (y: 3200, x: 3200)> Size: 82MB
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])
Coordinates:
    time         datetime64[ns] 8B 2020-07-01T23:59:59.999999
  * y            (y) float64 26kB -2.304e+06 -2.304e+06 ... -2.4e+06 -2.4e+06
  * x            (x) float64 26kB -1.344e+06 -1.344e+06 ... -1.248e+06
    spatial_ref  int32 4B 3577
Attributes:
    crs:      epsg:3577

In [31]:
write_cog(geo_im=all_burn_cleaned, fname="test.tif", overwrite=True, nodata=-999)

PosixPath('test.tif')

In [32]:
all_burn_cleaned

<xarray.DataArray (y: 3200, x: 3200)> Size: 82MB
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])
Coordinates:
    time         datetime64[ns] 8B 2020-07-01T23:59:59.999999
  * y            (y) float64 26kB -2.304e+06 -2.304e+06 ... -2.4e+06 -2.4e+06
  * x            (x) float64 26kB -1.344e+06 -1.344e+06 ... -1.248e+06
    spatial_ref  int32 4B 3577
Attributes:
    crs:      epsg:3577